In [0]:
'''1. Customer Purchase analysis with PySpark.
Calculate the total purchase amount for each customer

sample data
'''

from pyspark.sql import SparkSession
#from pyspark.sql import functions as F
#or
from pyspark.sql.functions import sum
spark = SparkSession.builder.appName("CustomerPurchaseAnalysis").getOrCreate()

data = [
    (1, 100, "2023-01-15"),
    (2, 150, "2023-02-20"),
    (1, 200, "2023-03-10"),
    (3, 50, "2023-04-05"),
    (2, 120, "2023-05-15"),
    (1, 300, "2023-06-25")
]
columns = ["customer_id", "purchase_amount", "purchase_date"]

df=spark.createDataFrame(data,columns)
df.show()

#DSL
df.groupBy("customer_id").agg(sum("purchase_amount").alias("Total_amt")).show()
#total_purchase_per_customer = df.groupBy("customer_id").agg(F.sum("purchase_amount").alias("total_purchase_amount"))
#SQL
df.createOrReplaceTempView("tbl")
ds_sql=spark.sql("select customer_id,sum(purchase_amount) as Total_amt from tbl group by customer_id")
ds_sql.show()

In [0]:
'''2.FIND THE CUSTOMER WITH THE HIGHEST
 TOTAL PURHASE AMOUNT

sample data'''

data = [
    (1, 100, "2023-01-15"),
    (2, 150, "2023-02-20"),
    (1, 200, "2023-03-10"),
    (3, 50, "2023-04-05"),
    (2, 120, "2023-05-15"),
    (1, 300, "2023-06-25")
]
columns = ["customer_id", "purchase_amount", "purchase_date"]
#DSL
df=spark.createDataFrame(data,columns)
df.show()
df1=df.groupBy("customer_id").agg(sum("purchase_amount").alias("Total_amt"))
df2=df1.orderBy(df1.Total_amt.desc())
customer_with_highest_purchase=df2.first()
print(customer_with_highest_purchase)
print("customer_with_highest_purchase is" ,customer_with_highest_purchase["customer_id"] ,"amount is ",customer_with_highest_purchase["Total_amt"])

print(
    f"Customer with highest purchase: {customer_with_highest_purchase['customer_id']} \n"
    f"Customer with highest Amount:  {customer_with_highest_purchase['Total_amt']} ")




In [0]:
'''
Question 2 using SQL
#SQL
'''

df.createOrReplaceTempView("tbl")
sql("select * from tbl").show()
df_sql=sql("select customer_id,sum(purchase_amount) as total_amt from tbl group by customer_id order by total_amt desc")
top_customer=df_sql.first()
print(top_customer)
#Preferred way
print(
    f"Customer with highest purchase: {top_customer['customer_id']} "
    f"| Total Amount: {top_customer['total_amt']}"
)
#if not using f string u have to convert to string
#print(top_customer["customer_id"]) - returning int. when we append with str, python wont accept
print("Customer ID: " +  str(top_customer["customer_id"]) + "Total_amt:" + str(top_customer['total_amt'])
      )





In [0]:
'''
3.Calculate the total revenue generated from all sales

sample data
'''
from pyspark.sql.functions import sum,col,avg
columns = ["product_id", "product_name", "category", "price", "quantity_sold"]
data = [
    (1, "Product A", "Electronics", 500, 100),
    (2, "Product B", "Clothing", 50, 200),
    (3, "Product C", "Electronics", 800, 50),
    (4, "Product D", "Beauty", 30, 300),
    (5, "Product E", "Clothing", 75, 150)
]

#DSL
df=spark.createDataFrame(data,columns)
df.show()
revenue = df.withColumn("revenue",col("price")*col("quantity_sold")).agg(sum("revenue").alias("total_revenue")).first()
print(revenue['total_revenue'])
#SQL
df.createOrReplaceTempView("sales_tbl")
spark.sql("select sum(price*quantity_sold) as revenue from sales_tbl").first()
print(revenue['total_revenue'])
#OR
spark.sql("select sum(price*quantity_sold) as revenue from sales_tbl").first()['revenue']
#Ref for first

df.createOrReplaceTempView("sales_tbl")

rows = spark.sql("select product_id, price from sales_tbl").first()
print(rows)
print(rows['product_id'])
print(rows['price'])


## Question 3.1: Top 5 best-selling products
#DSL
print('DSL Way')
df.orderBy(col("quantity_sold").desc()).show()

print('SQL Way')
df.createOrReplaceTempView("sales_tbl")
sql("select * from sales_tbl order by quantity_sold desc").show()



In [0]:

## Question 3.2 - Average price per category
print('DSL Way')
df.groupBy("category").agg(avg("price").alias("avg_price")).show()
print('SQL Way')
spark.sql("select avg(price) as avg_price,category from sales_tbl group by category").show()

In [0]:
# Question 4: Category with highest total revenue

data = [
    (1, "Product A", "Electronics", 500, 100),
    (2, "Product B", "Clothing", 50, 200),
    (3, "Product C", "Electronics", 800, 50),
    (4, "Product D", "Beauty", 30, 300),
    (5, "Product E", "Clothing", 75, 150)
]
columns = ["product_id", "product_name", "category", "price", "quantity_sold"]
#DSL
df=spark.createDataFrame(data,columns)

df.createOrReplaceTempView("sales_tbl")
#SQL
sql("select * from sales_tbl").show()
sql("select   category, sum(price*quantity_sold) as total_revenue from sales_tbl group by category order by total_revenue desc").show()
sql("select   category, sum(price*quantity_sold) as total_revenue from sales_tbl group by category order by total_revenue desc")

#DSL
df.show()
from pyspark.sql.functions import *
revenue_category=df.withColumn("total_revenue",col("price")*col("quantity_sold")).groupBy("category").agg(sum("total_revenue").alias("total_revenue"))
revenue_category.show()
max_revenue_category=revenue_category.orderBy(col("total_revenue").desc()).first()
print(f"category with highest revenue is {max_revenue_category['total_revenue']}")
 
 

In [0]:
#Qn 4:
'''
Dataset: The dataset is in CSV format and 
contains the following columns:
 employee_id, employee_name, department, salary.

Questions:

Calculate the total payroll cost for the company.

Find the average salary for each department.

Identify the highest-paid employee and their department.

Calculate the total number of employees in each department.

Sample Dataset:


'''

data = [
    (1, "John Doe", "Engineering", 90000),
    (2, "Jane Smith", "Marketing", 75000),
    (3, "Michael Johnson", "Engineering", 105000),
    (4, "Emily Davis", "Marketing", 80000),
    (5, "Robert Brown", "Engineering", 95000),
    (6, "Linda Wilson", "HR", 60000)
]
columns = ["employee_id", "employee_name", "department", "salary"]

emp_df=spark.createDataFrame(data,columns)
emp_df.show()

#Total payrol cost 
from pyspark.sql.functions import *
total_cost=emp_df.agg(sum("salary").alias("Total_cost"))
total_cost.show()
#AVG Sal for each dept
from pyspark.sql.functions import *
avg_sal_dept_wise_df=emp_df.groupBy("department").agg(avg("salary").alias("avg_cost"))
avg_sal_dept_wise_df.show()

#Identify the highest-paid employee and their department.
highest_paid_emp=emp_df.orderBy(col("salary").desc()).first()
print(highest_paid_emp)
print(f"employee is {highest_paid_emp['employee_name']} and highest paid salary is {highest_paid_emp['salary']}")
#Calculate the total number of employees in each department.
count_emp=emp_df.groupBy("department").count()
#OR
count_emp=emp_df.groupBy("department").agg(count("*").alias("emp_count"))
count_emp.show()
 
 

In [0]:
#Qn 5
#sample dataset
#Calculate the total number of orders for each customer.
#
data = [
    (1, "C101", "2023-07-01", 150),
    (2, "C102", "2023-07-02", 200),
    (3, "C101", "2023-07-02", 100),
    (4, "C103", "2023-07-03", 300),
    (5, "C102", "2023-07-04", 250),
    (6, "C101", "2023-07-05", 120)
]
columns = ["order_id", "customer_id", "order_date", "total_amount"]
order_df=spark.createDataFrame(data,columns)
order_df.show()
#DSL
from pyspark.sql.functions import *
total_orders_per_customer=order_df.groupBy("customer_id").agg(count("*"))
total_orders_per_customer.show()

In [0]:
#Qn 6 
'''

1: Average score per subject

2: Highest score and corresponding student per subject

3: Total number of students per subject

4: Subject(s) with the highest average score

sample dataset

'''
data = [
    (1, "Math", 85),
    (2, "Science", 92),
    (3, "Math", 78),
    (4, "English", 88),
    (5, "Science", 95),
    (6, "Math", 90)
]
columns = ["student_id", "subject", "score"]

student_df=spark.createDataFrame(data,columns)
student_df.show()
from pyspark.sql.functions import *
#Average score per subject
student_df1=student_df.groupBy("subject").agg(avg("score").alias("avg_score"))
student_df1.orderBy(col("avg_score").desc()).show()


#Highest score and corresponding student per subject
#using DSL
highest_score_per_subject = student_df.groupBy("subject").agg(max("score").alias("highest_score"))
highest_score_per_subject.show()
highest_score_students=student_df.join(highest_score_per_subject,on="subject",how="inner").filter(col("score")==col("highest_score"))
highest_score_students.show()

student_df.show()
student_df.createOrReplaceTempView("student")
high_score_df = spark.sql("select subject,max(score) as highest_score from student group by subject ")
high_score_df.show()
high_score_df.createOrReplaceTempView("high_score")
sql("select s.subject,s.student_id,s.score,h.highest_score from student s join high_score h where s.subject=h.subject and s.score=h.highest_score").show()
 


In [0]:
# Question 7

'''
7.Calculate the total revenue generated from all orders
	1: Total revenue generated from all orders
	2: Top 5 orders with highest total amount

sample data
'''


order_data = [
    (1, "C101", "2023-07-01", 150),
    (2, "C102", "2023-07-02", 200),
    (3, "C101", "2023-07-02", 100),
    (4, "C103", "2023-07-03", 300),
    (5, "C102", "2023-07-04", 250),
    (6, "C101", "2023-07-05", 120)
]
order_columns = ["order_id", "customer_id", "order_date", "total_amount"]

product_data = [
    (1, "Product A", 500),
    (2, "Product B", 50),
    (3, "Product C", 800),
    (4, "Product D", 30),
    (5, "Product E", 75)
]
product_columns = ["product_id", "product_name", "price"]

order_df=spark.createDataFrame(order_data,order_columns)
product_df=spark.createDataFrame(product_data,product_columns)
order_df.show()
product_df.show()

#Total revenue generated from all orders
order_df.selectExpr("sum(total_amount) as total_revenue").show()
#or
order_df.agg(sum("total_amount")).show()
row=order_df.selectExpr("sum(total_amount) as total_revenue").first()
print(row.total_revenue)



In [0]:
#Question 8
'''F𝐢𝐧𝐝 𝐭𝐡𝐞 𝐞𝐚𝐫𝐥𝐢𝐞𝐬𝐭 𝐚𝐧𝐝 𝐥𝐚𝐭𝐞𝐬𝐭 𝐭𝐢𝐦𝐞𝐬𝐭𝐚𝐦𝐩𝐬 𝐢𝐧 𝐭𝐡𝐞 𝐝𝐚𝐭𝐚𝐬𝐞𝐭
'''

from pyspark.sql.functions import *
columns = ["user_id", "timestamp"]
data = [
    ("user1", "2023-08-21 10:00:00"),
    ("user2", "2023-08-21 11:30:00"),
    ("user1", "2023-08-21 12:15:00"),
    ("user3", "2023-08-21 13:45:00"),
    ("user2", "2023-08-21 14:30:00"),
    ("user1", "2023-08-21 15:00:00")
]

df=spark.createDataFrame(data,columns)
df.show()
# 1. Find earliest and latest timestamps
df.printSchema()
new_df=df.withColumn("timestamp",col("timestamp").cast("timestamp"))
new_df.show()
#SQL
new_df.createOrReplaceTempView("temp")
spark.sql("select min(timestamp) as earliest,max(timestamp) as latest from temp").show()
#DSL
new_df.agg(min("timestamp").alias("earliest"),max("timestamp").alias("latest")).show()


# 2. Count the number of activities per user
activity_count = new_df.groupBy("user_id").agg(count("timestamp").alias("activity_count"))
activity_count.show()

#3.Calculate time duration between consecutive activities for each user
query1="""
select user_id,time1, previous_date,unix_timestamp(time1)-unix_timestamp(previous_date) as diff from (
select user_id,timestamp as time1,lag(timestamp) over(partition by user_id order by timestamp) previous_date  from temp t
)
"""
sql(query1).show()

#using DSL
window_spec = Window.partitionBy("user_id").orderBy("timestamp")
df2 = df2.withColumn("prev_timestamp", lag("timestamp").over(window_spec))
df2 = df2.withColumn("time_diff", (unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")).cast("int"))
df2.show()





In [0]:
# Qn. count the action
data = [
    (1, "login", "2023-08-20 10:23:45"),
    (2, "view", "2023-08-20 11:15:30"),
    (1, "purchase", "2023-08-20 12:45:18"),
    (3, "view", "2023-08-20 13:30:22")
]
columns = ["user_id", "action", "timestamp"]
df=spark.createDataFrame(data,columns)
df.show()

# Convert timestamp to timestamp type
df2 = df.withColumn("timestamp", col("timestamp").cast("timestamp"))
df2.count()
#Uniq actions in the dataset

df2.select (col("action")).distinct().show()

In [0]:
'''
11.Question 1: Calculate Average User Session Duration
You have a dataset containing user activity logs in a 
PySpark DataFrame with the following columns:
 user_id, timestamp, and action. 
The action column indicates whether the user
 started or ended a session. It can have values 
'start' or 'end'. Your task is to calculate the 
average duration of user sessions.

 Sample Data

'''

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, unix_timestamp, avg, when

# Initialize Spark Session
spark = SparkSession.builder.appName("AverageSessionDuration").getOrCreate()

# Sample Data
data = [
    (1, "2022-01-01 10:00", "start"),
    (1, "2022-01-01 10:15", "end"),
    (2, "2022-01-01 11:00", "start"),
    (1, "2022-01-01 11:30", "start"),
    (2, "2022-01-01 11:45", "end"),
    (1, "2022-01-01 12:00", "end"),
]
columns = ["user_id", "timestamp", "action"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert timestamp to proper TimestampType
df = df.withColumn("timestamp", col("timestamp").cast("timestamp"))
df.show()

# Define window partitioned by user_id and ordered by timestamp
window_spec= Window.partitionBy("user_id").orderBy("timestamp")

# Pair start and end actions: Use lag() to find the previous action and timestamp
df_with_lag=df.withColumn("previous_action",lag("action").over(window_spec)) \
    .withColumn("previous_timestamp",lag("timestamp").over(window_spec))
df_with_lag.show()   

# Filter for valid session pairs (start -> end)
valid_sessions = df_with_lag.filter((col("action") == "end") & (col("previous_action") == "start"))
valid_sessions.show()
